In [24]:
from fp_qsim import hello

output = hello()
print(output)

Hello from fp-qsim!


In [ ]:
# mocked simulator for testing purposes

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from dataclasses import dataclass
import numpy as np
from qiskit.quantum_info import Operator


@dataclass
class MockSimulator(AerSimulator):
	def __init__(self) -> None:
		super().__init__()
		self._statevector_simulator = AerSimulator(method='statevector')

	def run(self, circuits: QuantumCircuit, shots: int) -> object:  # type: ignore[invalid-method-override]
		result = self._statevector_simulator.run(circuits, shots=shots).result()
		return result


@dataclass
class CustomSimulatorManual:
	def apply_unitary(
		self, state: np.ndarray, gate_tensor: np.ndarray, qubits: list[int]
	) -> np.ndarray:
		n_qubits = state.ndim
		state_axes = list(range(n_qubits))[::-1]
		qubit_axes = list(qubits)

		out_axes = state_axes.copy()
		new_gate_out_axes = list(range(n_qubits, n_qubits + len(qubits)))
		for i, axis in enumerate(qubit_axes):
			out_axes[n_qubits - 1 - axis] = new_gate_out_axes[i]

		return np.einsum(
			gate_tensor,
			new_gate_out_axes + qubit_axes,
			state,
			state_axes,
			out_axes,
			optimize=False,
		)

	def apply_cx(self, state: np.ndarray, control: int, target: int) -> np.ndarray:
		"""Apply CX by swapping amplitudes where control=1 and target flips 0->1."""
		n_qubits = state.ndim
		flat = state.reshape(-1).copy()
		n_states = 1 << n_qubits

		for i in range(n_states):
			if ((i >> control) & 1) == 1 and ((i >> target) & 1) == 0:
				j = i | (1 << target)
				flat[i], flat[j] = flat[j], flat[i]

		return flat.reshape([2] * n_qubits)

	def run(self, circuit: QuantumCircuit, shots: int = 1024) -> np.ndarray:
		"""Apply each gate tensor to the state tensor."""
		circuit = transpile(circuit, basis_gates=['u', 'cx'])
		n_qubits = circuit.num_qubits

		# Tensor axes are ordered as [q_{n-1}, ..., q_0] so flattening matches Qiskit's basis order.
		state = np.zeros([2] * n_qubits, dtype=complex)
		state[(0,) * n_qubits] = 1.0

		for instruction in circuit.data:
			gate = instruction.operation
			qubits = [circuit.find_bit(q).index for q in instruction.qubits]
			if gate.name in [
				'measure',
				'barrier',
				'reset',
				'snapshot',
				'save_statevector',
				'load_statevector',
			]:
				continue
			elif gate.name == 'u':
				n_gate_qubits = len(qubits)
				# Qiskit's 'u' gate is a general single-qubit unitary, so we can directly use its matrix.
				gate_tensor = Operator(gate).data.reshape([2] * (2 * n_gate_qubits))
				state = self.apply_unitary(state, gate_tensor, qubits)
				continue
			elif gate.name == 'cx':
				control, target = qubits
				state = self.apply_cx(state, control, target)

		return state.reshape(-1)


ghz_circ = QuantumCircuit(2)
ghz_circ.h(0)
ghz_circ.cx(0, 1)
custom_state = CustomSimulatorManual().run(ghz_circ)

ref_sim = AerSimulator(method='statevector')
ghz_circ.save_statevector()
ref_state = ref_sim.run(ghz_circ).result().get_statevector()

print('Custom statevector:', custom_state)
print('Reference statevector:', ref_state)
print('Match:', np.allclose(custom_state, ref_state))

Custom statevector: [0.70710678+0.j 0.70710678+0.j 0.        +0.j 0.        +0.j]
Reference statevector: Statevector([0.70710678+0.j, 0.70710678+0.j, 0.        +0.j,
             0.        +0.j],
            dims=(2, 2))
Match: True


In [26]:
arr = np.array([1, 2, 3, 4])
print(arr.reshape(2, 2)[0, 1])

2
